In [149]:
import optuna
import numpy as np
import pandas as pd

from km_gg import GG_KM
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

In [150]:
lung_df = pd.read_csv("../data/lung.csv")\
    .drop(columns=["inst", "sex", "meal.cal", "wt.loss"])\
    .assign(status=lambda x: x.status.replace([1,2], [0,1]))\
    .fillna(0)

y = lung_df[["time", "status"]]
X = lung_df.drop(columns=y.columns.tolist()).to_numpy()

In [151]:
def kaplan_meier(t, delta):
    """Simple Kaplan-Meier estimator. Returns (times, survival)."""
    order    = np.argsort(t)
    t_ord    = t[order]
    d_ord    = delta[order]
    unique_t = np.unique(t_ord)

    surv = 1.0
    km_t = [0.0]
    km_s = [1.0]

    for ti in unique_t:
        events  = d_ord[t_ord == ti].sum()
        at_risk = (t_ord >= ti).sum()
        surv   *= 1 - events / at_risk
        km_t.append(ti)
        km_s.append(surv)

    return np.array(km_t), np.array(km_s)


def ipcw_weights(t_train, delta_train, t_eval, t_grid):
    """
    IPCW weights via Kaplan-Meier on the censoring distribution.
    weights[i, j] = 1 / G(min(t_eval[i], t_grid[j]))^2
    """
    km_times, km_surv = kaplan_meier(t_train, 1 - delta_train)

    def G(t_query):
        idx = np.searchsorted(km_times, t_query, side="right") - 1
        # idx = np.clip(idx, 0, len(km_surv) - 1)
        idx = idx
        # return np.maximum(km_surv[idx], 1e-4)   # floor avoids exploding weights
        return km_surv[idx]   # floor avoids exploding weights

    t_clip  = np.minimum(t_eval[:, None], t_grid[None, :])
    weights = 1.0 / G(t_clip) ** 2
    return weights


def integrated_brier_score(S_pred, t_eval, delta_eval,
                            t_train, delta_train, t_grid):
    """
    IBS with IPCW correction.

    Parameters
    ----------
    S_pred     : (m, T)  predicted survival probabilities on t_grid
    t_eval     : (m,)    observed times  (val/test set)
    delta_eval : (m,)    event indicators (val/test set)
    t_train    : (n,)    training times   (for censoring KM)
    delta_train: (n,)    training events
    t_grid     : (T,)    evaluation time grid

    Returns
    -------
    ibs : float
    """
    if not np.all(np.isfinite(S_pred)):
        return 1.0

    W = ipcw_weights(t_train, delta_train, t_eval, t_grid)

    event_before   = (t_eval[:, None] <= t_grid[None, :]) & (delta_eval[:, None] == 1)
    censored_after = t_eval[:, None] > t_grid[None, :]

    bs_grid = np.zeros(len(t_grid))
    for j in range(len(t_grid)):
        term1       = W[:, j] * event_before[:, j]   * S_pred[:, j] ** 2
        term2       = W[:, j] * censored_after[:, j]  * (1 - S_pred[:, j]) ** 2
        bs_grid[j]  = np.mean(term1 + term2)

    tau_min, tau_max = t_grid[0], t_grid[-1]
    ibs = np.trapezoid(bs_grid, t_grid) / (tau_max - tau_min)
    # return float(np.clip(ibs, 0.0, 1.0))
    return ibs


# =============================================================================
# Cross-Validation with Optuna
# =============================================================================

def cross_validate_gg_km(X, t, delta,
                          n_outer_splits=5,
                          n_inner_splits=4,
                          n_trials=50,
                          t_grid_points=50,
                          random_state=42):
    """
    Nested cross-validation for GG-KM with Optuna hyperparameter search.

    Structure
    ---------
    Outer loop : k-fold → each fold gives a held-out TEST set.
    Inner loop : k-fold CV on outer-train → Optuna minimizes mean val IBS.
    Final fit  : best hyperparams retrained on full outer-train, evaluated on TEST.
    """

    outer_cv        = KFold(n_splits=n_outer_splits, shuffle=True,
                            random_state=random_state)
    all_test_ibs    = []
    all_best_params = []

    print(f"{'='*60}")
    print(f" Nested CV: {n_outer_splits} outer × {n_inner_splits} inner folds")
    print(f" Optuna trials per fold: {n_trials}")
    print(f"{'='*60}\n")

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
        print(f"─── Outer fold {fold_idx + 1}/{n_outer_splits} ", end="", flush=True)

        X_outer_train, X_test = X[train_idx],        X[test_idx]
        t_outer_train, t_test = t[train_idx],        t[test_idx]
        d_outer_train, d_test = delta[train_idx],    delta[test_idx]

        # Time grid: use 5th–95th percentile of training times to avoid
        # extreme extrapolation that inflates IBS via IPCW weights
        t_lo   = np.percentile(t_outer_train, 5)
        t_hi   = np.percentile(t_outer_train, 95)
        t_grid = np.linspace(t_lo, t_hi, t_grid_points)

        # ----------------------------------------------------------
        # Inner loop: Optuna minimises mean validation IBS
        # ----------------------------------------------------------
        inner_cv = KFold(n_splits=n_inner_splits, shuffle=True,
                         random_state=random_state)

        def objective(trial):
            lambda_reg   = trial.suggest_float("lambda_reg",   1e-5, 1.0,  log=True)
            gamma_kernel = trial.suggest_float("gamma_kernel", 1e-3, 10.0, log=True)

            val_scores = []
            for tr_idx, val_idx in inner_cv.split(X_outer_train):
                X_tr,  X_val  = X_outer_train[tr_idx],  X_outer_train[val_idx]
                t_tr,  t_val  = t_outer_train[tr_idx],  t_outer_train[val_idx]
                d_tr,  d_val  = d_outer_train[tr_idx],  d_outer_train[val_idx]

                scaler  = StandardScaler()
                X_tr_s  = scaler.fit_transform(X_tr)
                X_val_s = scaler.transform(X_val)

                t_grid_inner = np.linspace(
                    np.percentile(t_tr, 5),
                    np.percentile(t_tr, 95),
                    t_grid_points
                )

                try:
                    model = GG_KM(lambda_reg=lambda_reg,
                                  gamma_kernel=gamma_kernel)
                    model.fit(X_tr_s, t_tr, d_tr)
                    S_pred = model.predict_survival(X_val_s, t_grid_inner)
                    ibs    = integrated_brier_score(
                        S_pred, t_val, d_val, t_tr, d_tr, t_grid_inner
                    )
                    val_scores.append(ibs)
                except Exception:
                    return 1.0   # penalise divergent trial

            return float(np.mean(val_scores))

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_params = study.best_params
        print(f"→ best val IBS = {study.best_value:.4f} | "
              f"λ={best_params['lambda_reg']:.2e}, "
              f"γ={best_params['gamma_kernel']:.3f}")

        # ----------------------------------------------------------
        # Refit on full outer-train with best hyperparameters
        # ----------------------------------------------------------
        scaler          = StandardScaler()
        X_outer_train_s = scaler.fit_transform(X_outer_train)
        X_test_s        = scaler.transform(X_test)

        final_model = GG_KM(
            lambda_reg   = best_params["lambda_reg"],
            gamma_kernel = best_params["gamma_kernel"],
        )
        final_model.fit(X_outer_train_s, t_outer_train, d_outer_train)

        S_test   = final_model.predict_survival(X_test_s, t_grid)
        test_ibs = integrated_brier_score(
            S_test, t_test, d_test, t_outer_train, d_outer_train, t_grid
        )

        print(f"           test  IBS = {test_ibs:.4f}\n")

        all_test_ibs.append(test_ibs)
        all_best_params.append(best_params)

    print(f"{'='*60}")
    print(f" Cross-Validation Results")
    print(f"{'='*60}")
    print(f" IBS per fold : {[f'{v:.4f}' for v in all_test_ibs]}")
    print(f" Mean IBS     : {np.mean(all_test_ibs):.4f}")
    print(f" Std  IBS     : {np.std(all_test_ibs):.4f}")
    print(f"{'='*60}\n")

    return {
        "test_ibs"    : all_test_ibs,
        "mean_ibs"    : float(np.mean(all_test_ibs)),
        "std_ibs"     : float(np.std(all_test_ibs)),
        "best_params" : all_best_params,
    }


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":

    lung_df = (
        pd.read_csv("../data/lung.csv")
        .drop(columns=["inst", "sex", "meal.cal", "wt.loss"])
        .assign(status=lambda x: x.status.replace([1, 2], [0, 1]))
        .fillna(0)
    )

    y     = lung_df[["time", "status"]]
    X_raw = lung_df.drop(columns=y.columns.tolist()).to_numpy()
    t     = y["time"].to_numpy().astype(float)
    delta = y["status"].to_numpy().astype(float)

    results = cross_validate_gg_km(
        X_raw, t, delta,
        n_outer_splits = 5,
        n_inner_splits = 4,
        n_trials       = 50,
        t_grid_points  = 50,
        random_state   = 42,
    )

def kaplan_meier(t, delta):
    """Simple Kaplan-Meier estimator. Returns (times, survival)."""
    order    = np.argsort(t)
    t_ord    = t[order]
    d_ord    = delta[order]
    unique_t = np.unique(t_ord)

    surv = 1.0
    km_t = [0.0]
    km_s = [1.0]

    for ti in unique_t:
        events  = d_ord[t_ord == ti].sum()
        at_risk = (t_ord >= ti).sum()
        surv   *= 1 - events / at_risk
        km_t.append(ti)
        km_s.append(surv)

    return np.array(km_t), np.array(km_s)


def ipcw_weights(t_train, delta_train, t_eval, t_grid):
    """
    IPCW weights via Kaplan-Meier on the censoring distribution.
    weights[i, j] = 1 / G(min(t_eval[i], t_grid[j]))^2
    """
    km_times, km_surv = kaplan_meier(t_train, 1 - delta_train)

    def G(t_query):
        idx = np.searchsorted(km_times, t_query, side="right") - 1
        # idx = np.clip(idx, 0, len(km_surv) - 1)
        idx = idx
        # return np.maximum(km_surv[idx], 1e-4)   # floor avoids exploding weights
        return km_surv[idx]   # floor avoids exploding weights

    t_clip  = np.minimum(t_eval[:, None], t_grid[None, :])
    weights = 1.0 / G(t_clip) ** 2
    return weights


def integrated_brier_score(S_pred, t_eval, delta_eval,
                            t_train, delta_train, t_grid):
    """
    IBS with IPCW correction.

    Parameters
    ----------
    S_pred     : (m, T)  predicted survival probabilities on t_grid
    t_eval     : (m,)    observed times  (val/test set)
    delta_eval : (m,)    event indicators (val/test set)
    t_train    : (n,)    training times   (for censoring KM)
    delta_train: (n,)    training events
    t_grid     : (T,)    evaluation time grid

    Returns
    -------
    ibs : float
    """
    if not np.all(np.isfinite(S_pred)):
        return 1.0

    W = ipcw_weights(t_train, delta_train, t_eval, t_grid)

    event_before   = (t_eval[:, None] <= t_grid[None, :]) & (delta_eval[:, None] == 1)
    censored_after = t_eval[:, None] > t_grid[None, :]

    bs_grid = np.zeros(len(t_grid))
    for j in range(len(t_grid)):
        term1       = W[:, j] * event_before[:, j]   * S_pred[:, j] ** 2
        term2       = W[:, j] * censored_after[:, j]  * (1 - S_pred[:, j]) ** 2
        bs_grid[j]  = np.mean(term1 + term2)

    tau_min, tau_max = t_grid[0], t_grid[-1]
    ibs = np.trapezoid(bs_grid, t_grid) / (tau_max - tau_min)
    # return float(np.clip(ibs, 0.0, 1.0))
    return float(ibs)


# =============================================================================
# Cross-Validation with Optuna
# =============================================================================

def cross_validate_gg_km(X, t, delta,
                          n_outer_splits=5,
                          n_inner_splits=4,
                          n_trials=50,
                          t_grid_points=50,
                          random_state=42):
    """
    Nested cross-validation for GG-KM with Optuna hyperparameter search.

    Structure
    ---------
    Outer loop : k-fold → each fold gives a held-out TEST set.
    Inner loop : k-fold CV on outer-train → Optuna minimizes mean val IBS.
    Final fit  : best hyperparams retrained on full outer-train, evaluated on TEST.
    """

    outer_cv        = KFold(n_splits=n_outer_splits, shuffle=True,
                            random_state=random_state)
    all_test_ibs    = []
    all_best_params = []

    print(f"{'='*60}")
    print(f" Nested CV: {n_outer_splits} outer × {n_inner_splits} inner folds")
    print(f" Optuna trials per fold: {n_trials}")
    print(f"{'='*60}\n")

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
        print(f"─── Outer fold {fold_idx + 1}/{n_outer_splits} ", end="", flush=True)

        X_outer_train, X_test = X[train_idx],        X[test_idx]
        t_outer_train, t_test = t[train_idx],        t[test_idx]
        d_outer_train, d_test = delta[train_idx],    delta[test_idx]

        # Time grid: use 5th–95th percentile of training times to avoid
        # extreme extrapolation that inflates IBS via IPCW weights
        t_lo   = np.percentile(t_outer_train, 5)
        t_hi   = np.percentile(t_outer_train, 95)
        t_grid = np.linspace(t_lo, t_hi, t_grid_points)

        # ----------------------------------------------------------
        # Inner loop: Optuna minimises mean validation IBS
        # ----------------------------------------------------------
        inner_cv = KFold(n_splits=n_inner_splits, shuffle=True,
                         random_state=random_state)

        def objective(trial):
            lambda_reg   = trial.suggest_float("lambda_reg",   1e-5, 1.0,  log=True)
            gamma_kernel = trial.suggest_float("gamma_kernel", 1e-3, 10.0, log=True)

            val_scores = []
            for tr_idx, val_idx in inner_cv.split(X_outer_train):
                X_tr,  X_val  = X_outer_train[tr_idx],  X_outer_train[val_idx]
                t_tr,  t_val  = t_outer_train[tr_idx],  t_outer_train[val_idx]
                d_tr,  d_val  = d_outer_train[tr_idx],  d_outer_train[val_idx]

                scaler  = StandardScaler()
                X_tr_s  = scaler.fit_transform(X_tr)
                X_val_s = scaler.transform(X_val)

                t_grid_inner = np.linspace(
                    np.percentile(t_tr, 5),
                    np.percentile(t_tr, 95),
                    t_grid_points
                )

                try:
                    model = GG_KM(lambda_reg=lambda_reg,
                                  gamma_kernel=gamma_kernel)
                    model.fit(X_tr_s, t_tr, d_tr)
                    S_pred = model.predict_survival(X_val_s, t_grid_inner)
                    ibs    = integrated_brier_score(
                        S_pred, t_val, d_val, t_tr, d_tr, t_grid_inner
                    )
                    val_scores.append(ibs)
                except Exception:
                    return 1.0   # penalise divergent trial

            return float(np.mean(val_scores))

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_params = study.best_params
        print(f"→ best val IBS = {study.best_value:.4f} | "
              f"λ={best_params['lambda_reg']:.2e}, "
              f"γ={best_params['gamma_kernel']:.3f}")

        # ----------------------------------------------------------
        # Refit on full outer-train with best hyperparameters
        # ----------------------------------------------------------
        scaler          = StandardScaler()
        X_outer_train_s = scaler.fit_transform(X_outer_train)
        X_test_s        = scaler.transform(X_test)

        final_model = GG_KM(
            lambda_reg   = best_params["lambda_reg"],
            gamma_kernel = best_params["gamma_kernel"],
        )
        final_model.fit(X_outer_train_s, t_outer_train, d_outer_train)

        S_test   = final_model.predict_survival(X_test_s, t_grid)
        test_ibs = integrated_brier_score(
            S_test, t_test, d_test, t_outer_train, d_outer_train, t_grid
        )

        print(f"           test  IBS = {test_ibs:.4f}\n")

        all_test_ibs.append(test_ibs)
        all_best_params.append(best_params)

    print(f"{'='*60}")
    print(f" Cross-Validation Results")
    print(f"{'='*60}")
    print(f" IBS per fold : {[f'{v:.4f}' for v in all_test_ibs]}")
    print(f" Mean IBS     : {np.mean(all_test_ibs):.4f}")
    print(f" Std  IBS     : {np.std(all_test_ibs):.4f}")
    print(f"{'='*60}\n")

    return {
        "test_ibs"    : all_test_ibs,
        "mean_ibs"    : float(np.mean(all_test_ibs)),
        "std_ibs"     : float(np.std(all_test_ibs)),
        "best_params" : all_best_params,
    }


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":

    lung_df = (
        pd.read_csv("../data/lung.csv")
        .drop(columns=["inst", "meal.cal", "wt.loss"])
        .assign(status=lambda x: x.status.replace([1, 2], [0, 1]))
        .fillna(0)
    )

    y     = lung_df[["time", "status"]]
    X_raw = lung_df.drop(columns=y.columns.tolist()).to_numpy()
    t     = y["time"].to_numpy().astype(float)
    delta = y["status"].to_numpy().astype(float)

    results = cross_validate_gg_km(
        X_raw, t, delta,
        n_outer_splits = 5,
        n_inner_splits = 4,
        n_trials       = 50,
        t_grid_points  = 50,
        random_state   = 42,
    )


 Nested CV: 5 outer × 4 inner folds
 Optuna trials per fold: 50

─── Outer fold 1/5 

[I 2026-05-29 17:18:52,162] A new study created in memory with name: no-name-9ceb5126-9357-409b-8a47-46d22bc4a0b9
[I 2026-05-29 17:18:53,071] Trial 0 finished with value: 0.5245878048757195 and parameters: {'lambda_reg': 0.0007459343285726547, 'gamma_kernel': 6.351221010640703}. Best is trial 0 with value: 0.5245878048757195.
[I 2026-05-29 17:18:54,226] Trial 1 finished with value: 0.3010823302967667 and parameters: {'lambda_reg': 0.0457056309980145, 'gamma_kernel': 0.24810409748678125}. Best is trial 1 with value: 0.3010823302967667.
[I 2026-05-29 17:18:55,263] Trial 2 finished with value: 0.2591109573043786 and parameters: {'lambda_reg': 6.0268891286825045e-05, 'gamma_kernel': 0.004207053950287938}. Best is trial 2 with value: 0.2591109573043786.
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:58: RuntimeWarning: invalid value encountered in scalar multiply
  def _log_fGG(self, t):
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicaca

→ best val IBS = 0.2476 | λ=3.03e-01, γ=0.004
           test  IBS = 0.1488

─── Outer fold 2/5 

[I 2026-05-29 17:19:32,478] A new study created in memory with name: no-name-a69ac9a3-296c-457c-9f91-73a1ad2ad45d
[I 2026-05-29 17:19:33,095] Trial 0 finished with value: 0.4427378633665207 and parameters: {'lambda_reg': 0.0007459343285726547, 'gamma_kernel': 6.351221010640703}. Best is trial 0 with value: 0.4427378633665207.
[I 2026-05-29 17:19:34,029] Trial 1 finished with value: 0.36839241456165417 and parameters: {'lambda_reg': 0.0457056309980145, 'gamma_kernel': 0.24810409748678125}. Best is trial 1 with value: 0.36839241456165417.
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:76: RuntimeWarning: overflow encountered in exp
  def _objective(self, params, K, t, delta):
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:81: RuntimeWarning: invalid value encountered in multiply
  # w = np.clip(np.exp(f), 0.0, _W_MAX)  # prevent overflow
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relato

→ best val IBS = 0.2886 | λ=3.65e-02, γ=0.002
           test  IBS = 0.3088

─── Outer fold 3/5 

...

→ best val IBS = 0.2382 | λ=1.95e-01, γ=0.002
           test  IBS = 0.2979

─── Outer fold 4/5 

[I 2026-05-29 17:24:26,103] A new study created in memory with name: no-name-96c769db-154a-42b0-9c59-067812c83975
[I 2026-05-29 17:24:26,699] Trial 0 finished with value: 0.6269904185844466 and parameters: {'lambda_reg': 0.0007459343285726547, 'gamma_kernel': 6.351221010640703}. Best is trial 0 with value: 0.6269904185844466.
[I 2026-05-29 17:24:27,860] Trial 1 finished with value: 0.32595395507835667 and parameters: {'lambda_reg': 0.0457056309980145, 'gamma_kernel': 0.24810409748678125}. Best is trial 1 with value: 0.32595395507835667.
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:59: RuntimeWarning: overflow encountered in power
  """Log of GG PDF, clipped for numerical safety."""
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:63: RuntimeWarning: overflow encountered in power
  np.log(p) + (d - 1) * np.log(t) - (t / a) ** p - d * np.log(a) - gammaln(s)
/home/cowvin/Documents/master_pipges/topicos_pesquis

→ best val IBS = 0.2543 | λ=5.72e-04, γ=0.001
           test  IBS = 0.2698

─── Outer fold 5/5 

[I 2026-05-29 17:25:08,472] A new study created in memory with name: no-name-42f548d5-cc19-4aa9-a284-b72768bb5d96
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:76: RuntimeWarning: overflow encountered in exp
  def _objective(self, params, K, t, delta):
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:58: RuntimeWarning: overflow encountered in scalar power
  def _log_fGG(self, t):
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:59: RuntimeWarning: overflow encountered in power
  """Log of GG PDF, clipped for numerical safety."""
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:59: RuntimeWarning: invalid value encountered in multiply
  """Log of GG PDF, clipped for numerical safety."""
/home/cowvin/Documents/master_pipges/topicos_pesquisa_I/aplicacao_relatorio_2/km_gg.py:81: RuntimeWarning: invalid value encountered in multiply
  # 

→ best val IBS = 0.2330 | λ=1.25e-01, γ=0.001
           test  IBS = 0.2407

 Cross-Validation Results
 IBS per fold : ['0.1614', '0.2842', '0.2979', '0.2698', '0.2407']
 Mean IBS     : 0.2508
 Std  IBS     : 0.0486

